# Public Dataset for Sentiments analysis

### Installing the dependencies

In [1]:
! pip uninstall -y torchvision # Ensure torchvision is completely removed
! pip uninstall -y datasets # Uninstall datasets to clear any cached state
! pip install transformers datasets torch scikit-learn

Found existing installation: torchvision 0.27.1
Uninstalling torchvision-0.27.1:
  Successfully uninstalled torchvision-0.27.1
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 1.2 MB/s  0:00:0036m-:--:--
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [datasets]2/3 [datasets]


### Import the library

In [4]:
#import the library

from transformers import BertTokenizer,BertForSequenceClassification,Trainer,TrainingArguments
from datasets import load_dataset
import torch
from sklearn.metrics import accuracy_score,precision_recall_fscore_support
import tqdm as notebook_tqdm

### LOAD THE ACTUAL DATASET

In [ ]:
#load the datset
dataset=load_dataset('stanfordnlp/imdb')
dataset=dataset.shuffle(seed=42)
small_train=dataset['train'].select(range(500))
small_test=dataset['test'].select(range(200))

In [4]:
print(small_test)

Dataset({
    features: ['text', 'label'],
    num_rows: 200
})


### Tokenisation


In [5]:
#Tokenize the dataset
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
def tokenizer_function(example):
  return tokenizer(example['text'],padding='max_length',truncation=True)
train_enc=small_train.map(tokenizer_function,batched=True)
test_enc=small_test.map(tokenizer_function,batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [6]:
train_enc

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})

### Creating tensors

In [7]:
#Prepare for Pytorch
train_enc.set_format('torch',columns=['input_ids','attention_mask','label'])
test_enc.set_format('torch',columns=['input_ids','attention_mask','label'])

### Model Selection / Loading Models

In [8]:
#loading the pretrained model for classifcation
model=BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Evaluation Function

In [9]:
#Define the evaluation metrics
def compute_metrics(pred):
  labels=pred.label_ids
  preds=pred.predictions.argmax(-1)
  labels=labels.astype('int')
  preds=preds.astype('int')
  precision,recall,f1,_=precision_recall_fscore_support(labels,preds,average='binary')
  acc=accuracy_score(labels,preds)
  return {
      'accuracy':float(acc),
      'f1':float(f1),
      'precision':float(precision),
      'recall':float(recall)
  }

In [10]:
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: transformers.trainer_utils.IntervalStrategy | str = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, gradient_accumulation_steps: int = 1, eval_accumulation_steps: int | None = None, eval_delay: float = 0, torch_empty_cache_steps: int | None = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_ratio: float | None = None, warmup_steps: float = 0, log_level: str = 'passive'

### Setting Up Training Config

In [11]:
#set training configuration
training_args=TrainingArguments(
    output_dir='.results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    num_train_epochs=2
)

### Initialising the trainer


In [12]:
#initialize the trainer
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    compute_metrics=compute_metrics
)

Model Training

In [13]:
#finetuning
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.454016,0.800000,0.816514,0.729508,0.927083
2,No log,0.361887,0.855000,0.857143,0.813084,0.906250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=126, training_loss=0.4361206236339751, metrics={'train_runtime': 113.4365, 'train_samples_per_second': 8.816, 'train_steps_per_second': 1.111, 'total_flos': 263111055360000.0, 'train_loss': 0.4361206236339751, 'epoch': 2.0})

### Model Evaluation

In [14]:
trainer.evaluate()

{'eval_loss': 0.36188697814941406,
 'eval_accuracy': 0.855,
 'eval_f1': 0.8571428571428571,
 'eval_precision': 0.8130841121495327,
 'eval_recall': 0.90625,
 'eval_runtime': 5.746,
 'eval_samples_per_second': 34.807,
 'eval_steps_per_second': 4.351,
 'epoch': 2.0}

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

inputs = tokenizer("Some input", return_tensors="pt").to(device)

# Now it's safe to run the model
outputs = model(**inputs)


In [16]:
#prediction on new text data
text='This is good Movie'
#inputs=tokenizer(text,return_tensors='pt',padding=True,truncation=True)
#outputs=model(**inputs)
prediction=torch.argmax(outputs.logits).item()
print('Predicted Sentiments','Positive' if prediction == 1 else 'Negative')


Predicted Sentiments Positive
